## Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "transfermarkt"

def gold(t):
    return spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/{t}")

fact_game = gold("fact_game")
fact_appearance = gold("fact_appearance")
fact_player_valuation = gold("fact_player_valuation")

## Label
Same win/draw/loss logic already validated in the EDA notebook's football sanity check.

In [0]:
fact_game_labeled = fact_game.withColumn(
    "result",
    F.when(F.col("home_club_goals") > F.col("away_club_goals"), "home_win")
     .when(F.col("home_club_goals") < F.col("away_club_goals"), "away_win")
     .otherwise("draw")
)
fact_game_labeled.groupBy("result").count().show()

## Rolling team form
Reshape to one row per team per game, then compute win rate over the 5 games strictly before each one. rowsBetween(-5, -1) is the leakage boundary — excluding the current row is what keeps this honest.

In [0]:
home_side = fact_game_labeled.select(
    "game_id", "date_key", F.col("home_club_key").alias("club_key"),
    F.when(F.col("result") == "home_win", 1).otherwise(0).alias("won"),
)
away_side = fact_game_labeled.select(
    "game_id", "date_key", F.col("away_club_key").alias("club_key"),
    F.when(F.col("result") == "away_win", 1).otherwise(0).alias("won"),
)
team_games = home_side.union(away_side)

form_window = Window.partitionBy("club_key").orderBy("date_key", "game_id").rowsBetween(-5, -1)

team_form = team_games.withColumn(
    "games_played_before", F.count("won").over(form_window)
).withColumn(
    "rolling_win_rate", F.avg("won").over(form_window)
)

print("Rows with fewer than 5 prior games (null or partial form):",
      team_form.filter(F.col("games_played_before") < 5).count(), "out of", team_form.count())

## Squad value at match time — as-of join
For each appearance, find the player's most recent valuation on or before that match's date. This can be a heavier join than the others — worth a glance at the Spark UI's stage timing once it runs, since it's a genuine as-of join rather than a simple equi-join.

In [0]:
appearance_dates = fact_appearance.select("appearance_id", "game_id", "player_key", "club_key", "date_key")

valuations_for_join = fact_player_valuation.select(
    F.col("player_key").alias("v_player_key"),
    F.col("date_key").alias("v_date_key"),
    "market_value_in_eur",
)

as_of = appearance_dates.join(
    valuations_for_join,
    (appearance_dates.player_key == valuations_for_join.v_player_key) &
    (valuations_for_join.v_date_key <= appearance_dates.date_key),
    "left",
)

latest_valuation_window = Window.partitionBy("appearance_id").orderBy(F.desc("v_date_key"))

player_value_at_match = (
    as_of
    .withColumn("rn", F.row_number().over(latest_valuation_window))
    .filter((F.col("rn") == 1) | F.col("v_date_key").isNull())
    .select("appearance_id", "game_id", "club_key", "market_value_in_eur")
)

squad_value = (
    player_value_at_match
    .groupBy("game_id", "club_key")
    .agg(
        F.sum("market_value_in_eur").alias("squad_value_eur"),
        F.count("*").alias("players_with_appearance"),
    )
)
squad_value.summary().show()
print("Games+clubs with zero appearances found:",
      fact_appearance.select("game_id", "club_key").distinct().count() - squad_value.count())

## Assemble the feature table
Home and away versions of both engineered features get joined onto the labeled games. home_club_goals/away_club_goals never make it into this table at all — only the derived result label does.

In [0]:
home_form = team_form.select(F.col("game_id"), F.col("club_key").alias("home_club_key"),
                              F.col("rolling_win_rate").alias("home_rolling_win_rate"),
                              F.col("games_played_before").alias("home_games_played_before"))
away_form = team_form.select(F.col("game_id"), F.col("club_key").alias("away_club_key"),
                              F.col("rolling_win_rate").alias("away_rolling_win_rate"),
                              F.col("games_played_before").alias("away_games_played_before"))

home_squad_value = squad_value.select(F.col("game_id"), F.col("club_key").alias("home_club_key"),
                                       F.col("squad_value_eur").alias("home_squad_value_eur"))
away_squad_value = squad_value.select(F.col("game_id"), F.col("club_key").alias("away_club_key"),
                                       F.col("squad_value_eur").alias("away_squad_value_eur"))

ml_features_classification = (
    fact_game_labeled
    .select("game_id", "result", "competition_key", "home_club_key", "away_club_key")
    .join(home_form, ["game_id", "home_club_key"], "left")
    .join(away_form, ["game_id", "away_club_key"], "left")
    .join(home_squad_value, ["game_id", "home_club_key"], "left")
    .join(away_squad_value, ["game_id", "away_club_key"], "left")
)

ml_features_classification.write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/classification_features")
print("Total rows:", ml_features_classification.count())
print("Rows with all 4 engineered features present:",
      ml_features_classification.dropna(subset=["home_rolling_win_rate", "away_rolling_win_rate",
                                                   "home_squad_value_eur", "away_squad_value_eur"]).count())
ml_features_classification.show(5)